# Build a Calendar + Lag-Feature Forecaster

**Week 7 · Lesson 6** &nbsp;·&nbsp; GBUS 564: Predictive Analytics &nbsp;·&nbsp; Fall 2026

<a target="_blank" href="https://colab.research.google.com/github/avorybryant/umw_gbus564_fall26/blob/main/notebooks/week7/lectures/06_calendar_lag_feature_forecaster.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

*Adapted from fpppy Ch.7 (Time series regression models), Ch.8–9 (Exponential smoothing; ARIMA models), and Ch.10 (Dynamic regression models) — the hands-on version of Lesson 5's regression reframe, raced against Lesson 4's classical models. Data: Australian Bureau of Statistics, catalogue 8501.0 (CC BY 4.0).*

## Setup

Same boilerplate as every notebook this term — see Week 1 Lesson 8 if this is
your first time seeing it.

In [ ]:
# Environment setup — identical across every notebook in the course.
# Installs the course package (dataset loaders + helpers) from the public
# course repo; Colab already has numpy/pandas/scikit-learn, so it's quick.
%pip install -q matplotlib git+https://github.com/avorybryant/umw_gbus564_fall26.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config

set_config(display="diagram")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_SEED = 564  # GBUS 564 — reuse this seed everywhere for reproducibility
rng = np.random.default_rng(RANDOM_SEED)

## 1. Business Problem

**Business question:** How much will Australians spend at cafes,
restaurants, and takeaway food outlets next month, so an operator or
supplier can plan staffing and inventory around it?

**Why it matters:** Understaffing during a high-demand month costs sales
and service quality; overstaffing during a slow month wastes a labor
budget that could go elsewhere (Lessons 7–8 and this week's assignment
return to this asymmetric-cost framing directly). Lesson 3 set the bar for any forecast that hopes to
inform this decision: it has to beat Seasonal Naive first.

## 2. Data

*Load, describe, and sanity-check the data before touching a model.*

**Australian retail trade turnover** (Australian Bureau of Statistics,
catalogue 8501.0 table 11; CC BY 4.0 — free to reuse and redistribute with
attribution), via `load_aus_retail()` — 152 parallel State × Industry
monthly series from 1982 onward. This notebook uses the national total for
**"Cafes, restaurants and takeaway food services"** (summed across all
eight states/territories), 2004–2018 — the same series fpppy's own Ch.10
dynamic-regression example uses.

**"Turnover" means sales.** It's the Australian/British word for revenue:
the total dollar value of what these businesses sold that month, in
millions of Australian dollars. (Not employee or inventory turnover.) So
every error below is in **$M of monthly sales**.

In [ ]:
from gbus564_coursework.utils.course_data import load_aus_retail

retail = load_aus_retail()
industry = retail[retail["Industry"] == "Cafes, restaurants and takeaway food services"]
series = industry.groupby("Month")["Turnover"].sum().loc["2004-01-01":"2018-12-01"]

df = pd.DataFrame({"sales": series.values}, index=series.index)
df.index.name = "month"

fig, ax = plt.subplots()
ax.plot(df.index, df["sales"], color="#1A73E8", linewidth=1.2)
ax.set_xlabel("Month"); ax.set_ylabel("Turnover ($M AUD)")
ax.set_title("Australia-wide cafe/restaurant/takeaway turnover, 2004-2018")
plt.tight_layout()

In [ ]:
df.head()

In [ ]:
df.describe()

### Data Prep

Three feature families, all from Lesson 5:

- **Trend** — `t`, the month number.
- **Calendar, in both encodings** — two Fourier pairs (smooth waves) and
  month dummies (11 on/off columns, January left out as the baseline), so
  Model Selection can let the data choose between them.
- **A menu of lags** — last month (`lag1`), two and three months back
  (`lag2`, `lag3`), the same month last year (`lag12`), the month before
  that a year ago (`lag13`), and the average of the last three months
  (`avg3`). Which of these earn a place is a Model Selection question.

Every lag is built with `.shift()`, so row *t* only ever sees months
*before* *t* — including the rolling average, which is shifted before it's
averaged. A lag or average that includes the current month would be
Lesson 2's leakage. Lags also cost history: `lag13` needs 13 earlier
months, so those rows are dropped — for **every** feature set, so all the
candidates below are compared on exactly the same rows.

In [ ]:
# Start from the raw turnover column every time, so this cell is safe to re-run
df = series.to_frame("sales").rename_axis("month")

df["t"] = np.arange(len(df))
for k in (1, 2):
    df[f"fourier_sin{k}"] = np.sin(2 * np.pi * k * df["t"] / 12)
    df[f"fourier_cos{k}"] = np.cos(2 * np.pi * k * df["t"] / 12)

months = pd.get_dummies(df.index.month, prefix="month", drop_first=True, dtype=float)
df = df.join(months.set_index(df.index))

for lag in (1, 2, 3, 12, 13):
    df[f"lag{lag}"] = df["sales"].shift(lag)
df["avg3"] = df["sales"].shift(1).rolling(3).mean()   # last 3 months, current month excluded

df = df.dropna()
print(f"{len(df)} rows remain after lagging (13 dropped from the start)")
df.head()

## 3. Model

*Baselines first — the reference every model must beat — then the models
under study.*

**Baselines** (Lesson 3): **Seasonal Naive** — this month's turnover is
the same month last year (already sitting in the `lag12` column) — and
**Naive** — this month is last month (`lag1`).

**Models under study:**

1. **Regression on calendar + lag columns** — Lesson 5's reframe, with the
   calendar encoding and the set of lags chosen together by a
   rolling-origin cross-validation sweep on the training months.
2. **XGBoost on the same columns** — the tree ensemble from Week 5, tuned on
   the same rolling-origin folds, to see
   how a flexible ML model handles a trending series.
3. **ETS and ARIMA** — Lesson 4's two classical families, each chosen
   automatically (the software picks the letters and fits the weights).

**How everything is scored.** Split **temporally**, not randomly (Lesson
2's rule): the last 20% of months (March 2016 – December 2018) are held
out as one contiguous block. The business question is *next month*, so
every model is scored **one month ahead**: for each held-out month, the
forecast uses everything up to the month before. The regression does this
naturally — its `lag1` column is last month's actual — and the classical
models do it with rolling-origin cross-validation (Lesson 2), one month at
a time. All models are fit once on the training months; after that, only new
data arrives, never new weights. (One honest asymmetry: ETS and ARIMA
train on all 146 pre-test months, while the regression and XGBoost lose
their first 13 to lags and train on 133 — a real advantage of models that
don't need lag columns, not an oversight.) And nothing is *chosen* by looking at
the held-out months — that's what the validation folds below are for.

### Model Selection

Which calendar encoding, and which lags? Treat both as **hyperparameters**
and sweep them — the standard way forecasters choose a lag set. Answer it
**without touching the held-out months**: choosing on the test block and
then reporting the test block as the final score would quietly leak
(Lesson 2). Instead, run rolling-origin cross-validation *inside* the
training months: scikit-learn's `TimeSeriesSplit` builds folds that always
train on the past and validate on the next stretch (here, 5 folds of 6
months each), exactly Lesson 2's rolling origin. Every combination of
calendar encoding × lag set gets the same folds; the lowest average
validation error wins.

In [ ]:
import itertools
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit

calendars = {
    "Fourier": ["t", "fourier_sin1", "fourier_cos1", "fourier_sin2", "fourier_cos2"],
    "Month dummies": ["t"] + [c for c in df.columns if c.startswith("month_")],
}
lag_sets = {
    "none": [],
    "1": ["lag1"],
    "12": ["lag12"],
    "1, 12": ["lag1", "lag12"],
    "1, 2, 12": ["lag1", "lag2", "lag12"],
    "1, 2, 3, 12": ["lag1", "lag2", "lag3", "lag12"],
    "1, 12, 13": ["lag1", "lag12", "lag13"],
    "1, 12, avg3": ["lag1", "lag12", "avg3"],
    "1, 2, 3, 12, 13, avg3": ["lag1", "lag2", "lag3", "lag12", "lag13", "avg3"],
}

cut = int(len(df) * 0.8)                 # last 20% = held-out test, untouched here
train_df = df.iloc[:cut]
# 5 folds, each validating on the next 6 months (test_size counts rows = months;
# "test" here means each fold's validation block, not the held-out test set).
# Each fold fits on every training month before its block, so the window grows.
folds = TimeSeriesSplit(n_splits=5, test_size=6)

def cv_mae(cols):
    scores = []
    for fit_rows, val_rows in folds.split(train_df):
        m = LinearRegression().fit(train_df[cols].iloc[fit_rows], train_df["sales"].iloc[fit_rows])
        scores.append(mean_absolute_error(train_df["sales"].iloc[val_rows],
                                          m.predict(train_df[cols].iloc[val_rows])))
    return np.mean(scores)

sweep = pd.DataFrame(
    [{"calendar": c, "lags": l, "CV MAE": cv_mae(calendars[c] + lag_sets[l])}
     for c, l in itertools.product(calendars, lag_sets)]
).sort_values("CV MAE").reset_index(drop=True)
print(sweep.round(1).to_string())

best = sweep.iloc[0]
feature_cols = calendars[best["calendar"]] + lag_sets[best["lags"]]
best_set = f"{best['calendar']} + lags {best['lags']}"
print(f"\nChosen: {best_set}")

Four things to read off the sweep:

- **Every lag at once won — here.** With month dummies, the full menu
  (lags 1, 2, 3, 12, 13 and the three-month average) beats every smaller
  set. That's 18 columns (plus an intercept) on 133 training months, which ordinary
  regression can afford; with less history or a bigger menu, the extra
  lags would start fitting noise. You don't have to guess which case
  you're in: the sweep scores both on months the model didn't train on.
- **Don't over-read the small gaps.** Below the winner, several sets sit
  within a point or two of each other. A sweep finds the *region* of good
  answers; take the winner and move on.
- **Calendar matters more than lags.** Fourier waves with lags 1 and 12
  sit far down the table, while month dummies with almost any lags sit
  near the top — Lesson 5's "dummies for short cycles."
- **Look at Fourier + lags 1, 12, 13.** It jumps nearly to the top. With
  the smooth Fourier calendar, last month's value carries last month's
  seasonal bump (January's `lag1` is December's spike); `lag13` is that
  same month a year earlier, the seasonal reference the regression needs
  to subtract the echo. That's Lesson 5's rule — pair a lag with the
  calendar of the period it came from — found by the search on its own.

In [ ]:
X, target = df[feature_cols], df["sales"]
X_train, X_test = X.iloc[:cut], X.iloc[cut:]
y_train, y_test = target.iloc[:cut], target.iloc[cut:]

# Baselines: Seasonal Naive (same month last year) and Naive (last month),
# already computed as the lag12 and lag1 columns. Read from df, not X_test,
# so they work whichever lags the sweep chose.
seasonal_naive_pred = df["lag12"].iloc[cut:]
naive_pred = df["lag1"].iloc[cut:]

model = LinearRegression().fit(X_train, y_train)
model_pred = model.predict(X_test)

print(f"Training rows: {len(X_train)}, test rows: {len(X_test)}")

### XGBoost: same columns, a tree model

Week 5's gradient-boosted trees, given exactly the regression's columns.
Those columns were chosen by the regression's sweep, not re-chosen for
trees — on purpose, so the comparison is about the model, not the
features. (A quick check shows the choice barely matters for XGBoost:
trees aren't confused by overlapping lags, and split month dummies or a
plain month number equally well.)

Two versions: one predicts **sales** directly; the other predicts the
**year-over-year change** (this month minus the same month last year), then
adds `lag12` back. Hyperparameters are chosen with the **same rolling-origin folds** as
the regression's sweep: a small grid over tree depth, learning rate, and
number of trees (12 combinations), searched separately for each version, so neither is
handicapped by settings tuned for the other. Each winner is then refit on
all the training months.

In [ ]:
%pip install -q xgboost
from xgboost import XGBRegressor

# Last year's value, read from df so this works whichever lags the sweep chose
last_year = df["lag12"]
offsets = {"sales": 0 * last_year,        # Version 1: predict sales directly
           "change": last_year}           # Version 2: predict sales - last year, add it back

# Tune each version on the same rolling-origin folds as the regression's sweep
def xgb_cv_mae(params, offset):
    target = y_train - offset.iloc[:cut]
    scores = []
    for fit_rows, val_rows in folds.split(X_train):
        m = XGBRegressor(**params, random_state=RANDOM_SEED).fit(
            X_train.iloc[fit_rows], target.iloc[fit_rows])
        pred = m.predict(X_train.iloc[val_rows]) + offset.iloc[:cut].iloc[val_rows].values
        scores.append(mean_absolute_error(y_train.iloc[val_rows], pred))
    return np.mean(scores)

grid = [dict(max_depth=d, learning_rate=lr, n_estimators=n)
        for d in (2, 3, 4) for lr in (0.03, 0.1) for n in (100, 300)]

xgb_preds = {}
for version, offset in offsets.items():
    sweep_v = pd.DataFrame([{**g, "CV MAE": round(xgb_cv_mae(g, offset), 1)} for g in grid])
    top = sweep_v.sort_values("CV MAE").iloc[0]
    params = dict(max_depth=int(top["max_depth"]), learning_rate=float(top["learning_rate"]),
                  n_estimators=int(top["n_estimators"]))
    print(f"Predicting {version}: best of {len(grid)} settings = depth {params['max_depth']}, "
          f"learning rate {params['learning_rate']}, {params['n_estimators']} trees "
          f"(CV MAE {top['CV MAE']:.1f})")
    # Refit the winning settings on all training months, then forecast the test months
    m = XGBRegressor(**params, random_state=RANDOM_SEED).fit(X_train, y_train - offset.iloc[:cut])
    xgb_preds[version] = m.predict(X_test) + offset.iloc[cut:].values

xgb_level_pred, xgb_pred = xgb_preds["sales"], xgb_preds["change"]

print(f"\nHighest sales in training: {y_train.max():,.0f}   highest in test: {y_test.max():,.0f}")
print(f"Highest XGBoost forecast -- predicting sales: {xgb_level_pred.max():,.0f}, "
      f"predicting the change: {xgb_pred.max():,.0f}")

# The flatline, up close: the sales-predicting version against the training ceiling
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(y_train.index, y_train, color="#5F6368", linewidth=1.0, label="Training months")
ax.plot(y_test.index, y_test, color="#202124", linewidth=1.6, label="Actual (test)")
ax.plot(y_test.index, xgb_level_pred, color="#D93025", linewidth=1.4, label="XGBoost, predicting sales")
ax.axhline(y_train.max(), color="#D93025", linestyle=":", linewidth=1, label="Highest training value")
ax.set_xlabel("Month"); ax.set_ylabel("Turnover ($M AUD)")
ax.legend(frameon=False, loc="upper left")
plt.tight_layout()


The sales-predicting version never forecasts above the highest value it
saw in training: a tree's prediction is an average of training rows that
landed in the same leaf, so it **can't extrapolate** past the training
range — the same flatline as Lesson 2's nearest-neighbor forecaster. On a
growing series, that's fatal. Predicting the *change* instead fixes it,
because year-over-year changes stay inside a range the trees have seen.

That fix is Lesson 4's seasonal difference — the **D** in ARIMA. Subtract
the same month last year, model the change, then undo the difference by
adding last year back: the same difference → model → undo pipeline, with
trees in the middle instead of MA terms. (The ARIMA below also takes a
first difference; this version doesn't.) That's why ML forecasters are
usually built on differences, ratios, or lags rather than raw levels.

The regression doesn't need this. Its trend column `t` already
extrapolates, and because `lag12` is one of its columns, predicting the
change would give identical forecasts — only `lag12`'s weight would shift
by exactly 1.

### The classical families: ETS and ARIMA

Lesson 4's models, fit with **statsforecast**, a forecasting library built
for exactly this. `AutoETS` searches the ETS forms (trend or not, damped or
not, additive or multiplicative season) and `AutoARIMA` picks the
differencing and then the AR/MA orders — both by AICc, the "software
decides" half of Lesson 4's two-families slide. `season_length=12` tells
them the cycle is a year of months.

statsforecast expects a "long" table with three columns: `unique_id`
(which series — handy when you forecast thousands at once), `ds` (the
date), and `y` (the value). `cross_validation` then runs Lesson 2's rolling
origin: `h=1` forecasts one month ahead, `n_windows=34` steps through each
held-out month, and `refit=False` fits once on the training months and
then only feeds in each new actual — the same rules the regression plays
by.

In [ ]:
%pip install -q statsforecast
from statsforecast import StatsForecast
from statsforecast.models import AutoETS, AutoARIMA

long = pd.DataFrame({"unique_id": "cafes", "ds": series.index, "y": series.values})
long["ds"] = pd.to_datetime(long["ds"])

sf = StatsForecast(models=[AutoETS(season_length=12), AutoARIMA(season_length=12)], freq="MS")
# Despite the name, this isn't k-fold CV. With n_windows=34 and refit=False, it fits
# each model once on the series minus its last 34 months (the 146 pre-test months),
# then steps through the 34 held-out months one at a time (h=1, step_size=1):
# each forecast uses the actuals up to the month before, with no refitting.
cv = sf.cross_validation(df=long, h=1, step_size=1, n_windows=len(y_test), refit=False)
cv = cv.set_index("ds")

# Same 34 held-out months as the regression
assert (cv.index == pd.to_datetime(y_test.index)).all()
ets_pred, arima_pred = cv["AutoETS"].values, cv["AutoARIMA"].values
cv.head()

## 4. Evaluate

*Score against the baseline, not in isolation — a metric only means
something next to what it's beating.*

In [ ]:
results = pd.Series({
    "Seasonal Naive": mean_absolute_error(y_test, seasonal_naive_pred),
    "Naive": mean_absolute_error(y_test, naive_pred),
    f"Regression ({best_set})": mean_absolute_error(y_test, model_pred),
    "XGBoost, predicting sales": mean_absolute_error(y_test, xgb_level_pred),
    "XGBoost, predicting the change": mean_absolute_error(y_test, xgb_pred),
    "ETS (auto)": mean_absolute_error(y_test, ets_pred),
    "ARIMA (auto)": mean_absolute_error(y_test, arima_pred),
}).sort_values()
baseline_mae = results["Seasonal Naive"]
print("One-month-ahead MAE on the held-out months ($M AUD):")
print(pd.DataFrame({"MAE": results.round(1),
                    "vs. Seasonal Naive": (results / baseline_mae - 1).map("{:+.0%}".format)}).to_string())

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(y_test.index, y_test, color="#202124", linewidth=1.6, label="Actual")
ax.plot(y_test.index, seasonal_naive_pred, color="#9AA0A6", linewidth=1.2,
        linestyle="--", label="Seasonal Naive")
ax.plot(y_test.index, model_pred, color="#1A73E8", linewidth=1.4, label="Regression")
ax.plot(y_test.index, xgb_pred, color="#9334E6", linewidth=1.2, label="XGBoost (change)")
ax.plot(y_test.index, ets_pred, color="#188038", linewidth=1.4, label="ETS (auto)")
ax.plot(y_test.index, arima_pred, color="#F9AB00", linewidth=1.4, label="ARIMA (auto)")
ax.set_xlabel("Month"); ax.set_ylabel("Turnover ($M AUD)")
ax.legend(frameon=False, ncol=3)
plt.tight_layout()

### Lesson 5's check: are the leftovers really leftover?

A model that used all the information in its training data leaves
residuals with no pattern — every ACF bar inside the shaded chance range.
The check isn't just for regressions: run it on every model you might
recommend. Four panels, all on the same 133 training months:

- **The regression's plain starting point** — Fourier waves with lags 1
  and 12.
- **The chosen regression.**
- **ETS and ARIMA** — actual minus each model's in-sample fitted value.
  (ARIMA needs 13 months of history before it can take both differences,
  so its first 13 months are warm-up; they're the same 13 months the
  regression lost to its lags.)

In [ ]:
%pip install -q statsmodels
from statsmodels.graphics.tsaplots import plot_acf

train_months = pd.to_datetime(y_train.index)

# The regressions' training leftovers
start_cols = calendars["Fourier"] + lag_sets["1, 12"]
start_model = LinearRegression().fit(train_df[start_cols], y_train)

# The classical models' training leftovers: fit both on the training months
# (the same fit the Interpret section reads labels from), then keep the
# residuals for the same months as the regression
train_long = long[long["ds"] < pd.to_datetime(y_test.index[0])]
sf.fit(train_long)
keep = train_long["ds"].isin(train_months).values
ets_resid = sf.fitted_[0, 0].model_["residuals"][keep]
arima_resid = sf.fitted_[0, 1].model_["residuals"][keep]

resid = {
    "Regression, starting point (Fourier + lags 1, 12)": y_train - start_model.predict(train_df[start_cols]),
    "Regression, chosen features": y_train - model.predict(X_train),
    "ETS (auto)": pd.Series(ets_resid),
    "ARIMA (auto)": pd.Series(arima_resid),
}
fig, axes = plt.subplots(2, 2, figsize=(12, 7.5), sharey=True)
for ax, (name, r) in zip(axes.flat, resid.items()):
    plot_acf(r.values, lags=24, ax=ax, title=name)
    ax.set_xlabel("Lag (months)")
plt.tight_layout()


The starting point fails badly — bars well outside the chance range at
lag after lag. The chosen regression passes: the few bars that poke out
are scattered, about what chance alone gives across 24 lags.

That's Lesson 5's loop — fit, check, diagnose, re-engineer, check again —
and the sweep already ran it. The starting point's leftover pattern came
from its smooth calendar missing the sharp December spike, and the sweep
found **two cures**: a better calendar (month dummies), or giving last
month's value its seasonal reference (`lag13`). Either way the leftovers
go quiet, and no second-stage fix is needed.

**ETS and ARIMA pass too** — a stray bar or two near the edge, no run of
bars outside the band. That matters for more than tidiness: the
recommended model's leftovers being patternless is what lets Lesson 8
trust the forecast ranges built on top of it.

### Can trees help ARIMA?

XGBoost lost the race on its own — but a tree doesn't have to replace
ARIMA. It can *correct* it. ARIMA's leftovers passed the check above: no
pattern left in time, lag by lag. But the ACF only looks for patterns in
time. A month with five Saturdays instead of four, or Easter landing in
March one year and April the next, moves café turnover in ways no lag of
the series can see, and ARIMA has no column for them.

So give a tree exactly that job: predict **ARIMA's miss** each month from
its recent misses and a few calendar facts — days in the month, weekend
days, trading days, and whether Easter falls in it — then add the
predicted correction back to ARIMA's forecast. Same rules as everything
else: settings chosen on the training months' rolling-origin folds, the
same 12-setting grid as the XGBoost above, scored once on the held-out
months.

In [ ]:
# The hybrid: ARIMA makes the forecast; a small XGBoost model predicts ARIMA's miss
# (actual - forecast) and adds it back.  Final forecast = ARIMA + predicted miss.
from dateutil.easter import easter

def calendar_facts(months):
    """Calendar facts a lag can't carry: days, weekend days, trading days, Easter."""
    months = pd.to_datetime(pd.Index(months))
    # Every calendar day in each month, so weekdays can be counted
    days = [pd.date_range(m, m + pd.offsets.MonthEnd(0)) for m in months]
    weekend = [int((d.dayofweek >= 5).sum()) for d in days]      # Saturdays + Sundays
    return pd.DataFrame({"month_num": months.month,
                         "days": [len(d) for d in days],
                         "weekend_days": weekend,
                         "trading_days": [len(d) - w for d, w in zip(days, weekend)],
                         "easter": [int(easter(m.year).month == m.month) for m in months]},
                        index=months)

# ---- The tree's target: ARIMA's miss each month ----
# ARIMA's misses: training months from the fit above, held-out months from its
# one-month-ahead forecasts (the same forecasts scored in the race)
test_months = pd.to_datetime(y_test.index)
miss = pd.concat([pd.Series(arima_resid, index=train_months),
                  pd.Series(y_test.values - arima_pred, index=test_months)])

# ---- The tree's inputs: calendar facts + ARIMA's recent misses ----
hyb = calendar_facts(miss.index)
for k in (1, 2, 3, 12):
    # Last month's miss is known once last month's actual arrives, so these are
    # all available when this month's forecast is made -- no peeking ahead
    hyb[f"miss_lag{k}"] = miss.shift(k)          # ARIMA's recent misses, known by forecast time
hyb["miss"] = miss
hyb = hyb.dropna()                               # first 12 months have no miss_lag12
hyb_cols = [c for c in hyb.columns if c != "miss"]
hyb_train, hyb_test = hyb[hyb.index < test_months[0]], hyb.loc[test_months]

# ---- Choose the tree's settings without touching the test months ----
# Same grid and rolling-origin folds as the XGBoost above, on the training months only
def hyb_cv_mae(params):
    scores = []
    for fit_rows, val_rows in folds.split(hyb_train):
        m = XGBRegressor(**params, random_state=RANDOM_SEED).fit(
            hyb_train[hyb_cols].iloc[fit_rows], hyb_train["miss"].iloc[fit_rows])
        scores.append(mean_absolute_error(hyb_train["miss"].iloc[val_rows],
                                          m.predict(hyb_train[hyb_cols].iloc[val_rows])))
    return np.mean(scores)

hyb_params = min(grid, key=hyb_cv_mae)          # the setting with the lowest CV MAE

# ---- Fit once on training months, then correct ARIMA's held-out forecasts ----
corrector = XGBRegressor(**hyb_params, random_state=RANDOM_SEED).fit(hyb_train[hyb_cols], hyb_train["miss"])
hybrid_pred = arima_pred + corrector.predict(hyb_test[hyb_cols])   # ARIMA + predicted miss

print(f"Corrector settings: depth {hyb_params['max_depth']}, learning rate {hyb_params['learning_rate']}, "
      f"{hyb_params['n_estimators']} trees")
print(f"ARIMA alone:           MAE {mean_absolute_error(y_test, arima_pred):.1f}")
print(f"ARIMA + XGBoost fix:   MAE {mean_absolute_error(y_test, hybrid_pred):.1f}  ($M AUD)")

# What the corrector uses most (share of its splitting gain)
print("\nWhat the corrector uses most:")
print(pd.Series(corrector.feature_importances_, index=hyb_cols).sort_values(ascending=False)
      .round(2).head(5).to_string())

The correction takes ARIMA from \$43.1M to \$39.7M — the best number in
this notebook, about 8% better than ARIMA alone. And look at what the tree
keys on: **trading days and weekend days** top the list. A month with one
more weekend, or one fewer trading day, shifts café turnover in a way no
lag of the series can carry — and ARIMA's leftovers looked clean in the
check above only because the ACF never looks there.

That's how trees most often earn their place in forecasting: not by
replacing the classical model, but by correcting what it can't see. Two
caveats. The corrector learned from ARIMA's misses on its own training
months, which run a little smaller than its true forecast misses; the
held-out score is still honest, because there it corrects ARIMA's real
one-month-ahead forecasts. And it's one test block — read the gain as
modest, not guaranteed.

## 5. Interpret

*What is the model actually keying off of? Business-readable, not just
statistically significant.*

Interpret the models the recommendation points to: the classical pair.
Weeks 3 and 5 covered reading a regression's coefficients and a tree's
importances. A classical forecaster is read differently: through its
label, and through the pieces its forecast is built from.

### Reading the classical models' labels

Lesson 4's skill: read the fitted label and say what it assumes about the
series. Both models were already fit on the training months for the residual check;
print what the automatic search chose.

In [ ]:
from statsforecast.arima import arima_string


print("ETS chose:  ", sf.fitted_[0, 0].model_["method"])
print("ARIMA chose:", arima_string(sf.fitted_[0, 1].model_))

- **ETS(A,N,A)** — additive errors, **no** separate trend, an additive
  seasonal pattern. The growth is handled by the level itself drifting
  upward month by month, and the December spike is an additive seasonal
  bump — measured in dollars, not percent — that the model keeps updating
  as new years arrive.
- **ARIMA(0,1,1)(0,1,1)[12]** — one first difference and one seasonal
  difference (the I and the D), then one MA term at lag 1 and one at the
  seasonal lag (the q and the Q). No AR terms at all. This exact model is
  famous enough to have a name, the "airline model," after the airline
  passenger series it was first fit to — and it's the same seasonal
  structure Lesson 4's scoreboard picked.

### What ETS's forecast is made of

The label says ETS(A,N,A) builds every forecast from two pieces: a
**level**, and a **seasonal bump** for each calendar month. Both are
running estimates that update as each month arrives, so any forecast can
be taken apart into numbers an operator can check.

In [ ]:
# ETS(A,N,A) carries two running estimates and updates both every month:
#   level -- where turnover sits with the season taken out
#   bump  -- each calendar month's usual amount above or below that level
# Its next-month forecast is just: the latest level + that month's latest bump.
ets_fit = sf.fitted_[0, 0].model_                  # the training-months fit from the residual check
states = ets_fit["states"]                         # a starting row, then one row per training month
fit_months = pd.to_datetime(train_long["ds"]).values
level = pd.Series(states[1:, 0], index=fit_months)  # the level after each month arrived
bump = pd.Series(states[1:, 1], index=fit_months)   # that calendar month's bump, just updated

alpha, gamma = ets_fit["par"][0], ets_fit["par"][2]
print(f"Each month's surprise moves the level {alpha:.0%} of the way, "
      f"and that month's bump {gamma:.0%} of the way")

# The first held-out forecast, taken apart
first = pd.Timestamp(y_test.index[0])
latest_bump = bump[bump.index.month == first.month].iloc[-1]
print(f"{first:%B %Y} forecast = level ${level.iloc[-1]:,.0f}M + {first:%B} bump "
      f"${latest_bump:+,.0f}M = ${level.iloc[-1] + latest_bump:,.0f}M "
      f"(ETS's own forecast: ${ets_pred[0]:,.0f}M)")

# December's bump, year by year, against the single fixed value a month dummy would use
december = bump[bump.index.month == 12]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.plot(train_long["ds"], train_long["y"], color="#5F6368", linewidth=0.8, alpha=0.6, label="Actual")
ax1.plot(level.index, level, color="#188038", linewidth=2, label="ETS level (season removed)")
ax1.set_title("The level: growth, with the season taken out")
ax1.set_xlabel("Month"); ax1.set_ylabel("Turnover ($M AUD)")
ax1.legend(frameon=False)
ax2.bar(december.index.year, december.values, color="#188038")
ax2.axhline(december.mean(), color="#202124", linestyle=":", linewidth=1.2,
            label="One fixed bump (what a month dummy assumes)")
ax2.set_title("December's bump: updated every year")
ax2.set_xlabel("Year"); ax2.set_ylabel("December above the level ($M AUD)")
ax2.legend(frameon=False, loc="upper left")
plt.tight_layout()
print(f"December bump: ${december.iloc[0]:,.0f}M in {december.index[0].year}, "
      f"${december.iloc[-1]:,.0f}M in {december.index[-1].year}")

**Reading it.** March 2016's forecast is \$3,428M of level plus March's
usual \$71M bump: exactly ETS's own number, and a sentence an operator can
check against their sense of the business. That's the recommendation's
practicality argument, made concrete.

The right panel is why the classical models won the race. ETS's December
bump grew from \$189M above the level in 2004 to \$441M in 2015, because
each December's surprise moves that month's bump 42% of the way. A month
dummy gives every year the same amount, like the dotted line: too big in
the early years, too small by the end, right before the test months. The
regression's `lag12` and `lag13` cover for that only partly.

**And the hybrid?** Its corrector is a tree, read the Week 5 way:
through the importances printed with it in Evaluate. Trading days and
weekend days come out on top, the calendar facts ARIMA has no column for.

## 6. Recommend

**Recommendation:** Use one of the **classical models** for next-month
planning. Both automatic ETS and ARIMA cut Seasonal Naive's error by about
two-thirds on held-out months, ahead of everything else here. They land
close together, so pick on practicality: ETS is the easier one to explain
to an operator ("this month's level, plus the usual seasonal bump for this
month"). If the last few million matter, let a small XGBoost correct
ARIMA's misses from calendar facts — the best result here.

- **Why the classical models win here** — their seasonal pattern
  *updates*; the regression's doesn't. Month dummies are fixed dollar
  amounts averaged over every training year, but this series' December
  bump grows with the business: about $190M above November in 2004–07,
  about $345M by 2016–18 (steady near 10% in percentage terms). ETS and
  ARIMA re-estimate the level and the season every month, weighting recent
  years more, so they follow that drift; the regression only partly
  catches it, through `lag12` and `lag13`. fpppy names this as calendar
  regression's one real weakness against seasonal ARIMA: "the seasonality
  is assumed to be fixed — the seasonal pattern is not allowed to change
  over time" (Ch.10). Fewer parameters help too — the ARIMA here has two,
  the regression 18 (plus an intercept), fit on 133 months. The gap is modest,
  though: one test block, one month ahead.

**What the other models taught:**

- **The regression** — with the calendar and lags chosen by a
  rolling-origin CV sweep, month dummies beat the smooth Fourier waves
  (Lesson 5's "dummies for short cycles"), the full menu of lags beats every
  smaller set,
  and the residuals pass the check. It beats Seasonal Naive by more than
  half, but still trails the classical models.
- **XGBoost** — predicting sales directly, it's *worse than Seasonal
  Naive*: trees can't forecast above anything they saw in training, and
  this series keeps growing. Predicting the year-over-year change instead
  brings it level with the regression — a hair ahead on this test block,
  well within noise. Flexible doesn't mean better at
  forecasting; how you frame the target matters more.
- **Trees as ARIMA's partner** — asked to predict ARIMA's *misses* rather
  than sales, a small XGBoost cut ARIMA's error from \$43.1M to \$39.7M,
  keying mostly on trading days and weekend days — calendar facts ARIMA has
  no column for. Same framing lesson one level up: the job you give the
  tree matters more than the tree.
- **The residual check** — it flagged the plain Fourier regression; the
  sweep had already found two cures (month dummies, or a `lag13`
  reference), and the chosen model passes.

**Caveats / next steps:**

- **One month ahead only.** Every number here is a next-month forecast fed
  last month's actual. A twelve-month plan is a harder problem, and the
  gaps between models would change (Lesson 7 returns to error by horizon).
- MAE stood in for Lesson 7's scaled metric (MASE); XGBoost's tuning grid was small
  (12 settings), though a broader search with more features moved it only about
  \$2M on its own;
  and one held-out block is one test, not a guarantee. Lesson 8 adds forecast ranges
  instead of single numbers, and Lesson 9 more flexible models.